In [3]:
# ============================================================
# CONFIGURAÇÃO DO PROJETO
# ============================================================

from pathlib import Path

BASE_DIR = Path(
    r"C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao"
)


print("BASE_DIR:")
print(BASE_DIR)


print("\nDiretório existe:")
print(BASE_DIR.exists())


print("\nPasta data existe:")
print(
    (
        BASE_DIR
        / "data"
    ).exists()
)

BASE_DIR:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao

Diretório existe:
True

Pasta data existe:
True


In [4]:
# ============================================================
# NOTEBOOK 10
# INTEGRAÇÃO FINAL AGROAMBIENTAL
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Caminhos
# ------------------------------------------------------------

ARQUIVO_BASE_AGROAMBIENTAL = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "integracao_agroambiental"
    / "base_agroambiental_pam_solo_inmet_cobertura_soja_centro_oeste_sul_2019_2024.csv"
)


ARQUIVO_SEEG = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "seeg"
    / "municipio_ano"
    / "seeg_emissoes_soja_municipio_ano_2019_2024.csv"
)


ARQUIVO_BRLUC = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "embrapa_brluc"
    / "municipio"
    / "brluc_soja_consolidado_centro_oeste_sul_2000_2019.csv"
)


# ------------------------------------------------------------
# Carregamento
# ------------------------------------------------------------

base_agroambiental = pd.read_csv(
    ARQUIVO_BASE_AGROAMBIENTAL,
    dtype={
        "codigo_ibge": "string"
    },
    encoding="utf-8-sig"
)


seeg = pd.read_csv(
    ARQUIVO_SEEG,
    dtype={
        "codigo_ibge": "string"
    },
    encoding="utf-8-sig"
)


brluc = pd.read_csv(
    ARQUIVO_BRLUC,
    dtype={
        "codigo_ibge": "string"
    },
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# Garantir código IBGE com 7 dígitos
# ------------------------------------------------------------

for base in [
    base_agroambiental,
    seeg,
    brluc
]:

    base[
        "codigo_ibge"
    ] = (
        base[
            "codigo_ibge"
        ]
        .str.zfill(7)
    )


print("Base agroambiental:", base_agroambiental.shape)
print("SEEG:", seeg.shape)
print("BRLUC:", brluc.shape)

Base agroambiental: (8674, 45)
SEEG: (9948, 26)
BRLUC: (1658, 74)


In [5]:
# ============================================================
# AUDITORIA DAS CHAVES
# ============================================================

print("=" * 70)
print("BASE AGROAMBIENTAL")
print("=" * 70)

print(
    "Municípios:",
    base_agroambiental[
        "codigo_ibge"
    ].nunique()
)

print(
    "Anos:",
    sorted(
        base_agroambiental[
            "ano"
        ]
        .unique()
    )
)

print(
    "Duplicatas codigo_ibge + ano:",
    base_agroambiental
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


print("\n" + "=" * 70)
print("SEEG")
print("=" * 70)

print(
    "Municípios:",
    seeg[
        "codigo_ibge"
    ].nunique()
)

print(
    "Anos:",
    sorted(
        seeg[
            "ano"
        ]
        .unique()
    )
)

print(
    "Duplicatas codigo_ibge + ano:",
    seeg
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


print("\n" + "=" * 70)
print("BRLUC")
print("=" * 70)

print(
    "Municípios:",
    brluc[
        "codigo_ibge"
    ].nunique()
)

print(
    "Duplicatas codigo_ibge:",
    brluc
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)

print(
    "Período BRLUC:",
    brluc[
        "periodo_inicio_brluc"
    ].unique(),
    "→",
    brluc[
        "periodo_fim_brluc"
    ].unique()
)

BASE AGROAMBIENTAL
Municípios: 1505
Anos: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Duplicatas codigo_ibge + ano: 0

SEEG
Municípios: 1658
Anos: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Duplicatas codigo_ibge + ano: 0

BRLUC
Municípios: 1658
Duplicatas codigo_ibge: 0
Período BRLUC: [2000] → [2019]


In [6]:
# ============================================================
# COBERTURA DAS FONTES SOBRE A BASE ANALÍTICA
# ============================================================

CHAVES_BASE = (
    base_agroambiental[
        [
            "codigo_ibge",
            "ano"
        ]
    ]
    .drop_duplicates()
)


CHAVES_SEEG = (
    seeg[
        [
            "codigo_ibge",
            "ano"
        ]
    ]
    .drop_duplicates()
)


CODIGOS_BRLUC = set(
    brluc[
        "codigo_ibge"
    ]
)


# ------------------------------------------------------------
# Cobertura SEEG
# ------------------------------------------------------------

auditoria_seeg = (
    CHAVES_BASE
    .merge(
        CHAVES_SEEG,
        on=[
            "codigo_ibge",
            "ano"
        ],
        how="left",
        indicator=True,
        validate="one_to_one"
    )
)


print("=" * 70)
print("COBERTURA SEEG SOBRE A BASE ANUAL")
print("=" * 70)

display(
    auditoria_seeg[
        "_merge"
    ]
    .value_counts()
)


# ------------------------------------------------------------
# Cobertura BRLUC
# ------------------------------------------------------------

auditoria_brluc = (
    base_agroambiental[
        [
            "codigo_ibge"
        ]
    ]
    .drop_duplicates()
    .copy()
)


auditoria_brluc[
    "possui_brluc"
] = (
    auditoria_brluc[
        "codigo_ibge"
    ]
    .isin(
        CODIGOS_BRLUC
    )
)


print(
    "\n" + "=" * 70
)

print(
    "COBERTURA BRLUC SOBRE OS MUNICÍPIOS DA BASE"
)

print("=" * 70)


display(
    auditoria_brluc[
        "possui_brluc"
    ]
    .value_counts()
)


print(
    "\nMunicípios sem BRLUC:"
)

display(
    auditoria_brluc[
        ~auditoria_brluc[
            "possui_brluc"
        ]
    ]
)

COBERTURA SEEG SOBRE A BASE ANUAL


_merge
both          8674
left_only        0
right_only       0
Name: count, dtype: int64


COBERTURA BRLUC SOBRE OS MUNICÍPIOS DA BASE


possui_brluc
True    1505
Name: count, dtype: int64


Municípios sem BRLUC:


,codigo_ibge,possui_brluc


In [7]:
# ============================================================
# AUDITORIA DE COLUNAS SOBREPOSTAS
# ============================================================

colunas_base = set(
    base_agroambiental.columns
)

colunas_seeg = set(
    seeg.columns
)

colunas_brluc = set(
    brluc.columns
)


sobreposicao_base_seeg = sorted(
    colunas_base
    .intersection(
        colunas_seeg
    )
)


sobreposicao_base_brluc = sorted(
    colunas_base
    .intersection(
        colunas_brluc
    )
)


sobreposicao_seeg_brluc = sorted(
    colunas_seeg
    .intersection(
        colunas_brluc
    )
)


print("=" * 70)
print("BASE × SEEG")
print("=" * 70)

print(
    sobreposicao_base_seeg
)


print(
    "\n" + "=" * 70
)

print(
    "BASE × BRLUC"
)

print("=" * 70)

print(
    sobreposicao_base_brluc
)


print(
    "\n" + "=" * 70
)

print(
    "SEEG × BRLUC"
)

print("=" * 70)

print(
    sobreposicao_seeg_brluc
)

BASE × SEEG
['ano', 'codigo_ibge', 'cultura', 'municipio', 'uf']

BASE × BRLUC
['codigo_ibge', 'cultura', 'municipio', 'regiao', 'uf']

SEEG × BRLUC
['codigo_ibge', 'cultura', 'fonte', 'municipio', 'uf']


In [8]:
# ============================================================
# INVENTÁRIO DE COLUNAS DAS FONTES A INTEGRAR
# ============================================================

print("=" * 70)
print("COLUNAS SEEG")
print("=" * 70)

for i, coluna in enumerate(
    seeg.columns,
    start=1
):

    print(
        f"{i:02d} -> {coluna}"
    )


print(
    "\n" + "=" * 70
)

print(
    "COLUNAS BRLUC"
)

print("=" * 70)


for i, coluna in enumerate(
    brluc.columns,
    start=1
):

    print(
        f"{i:02d} -> {coluna}"
    )

COLUNAS SEEG
01 -> codigo_ibge
02 -> municipio
03 -> uf
04 -> ano
05 -> cultura
06 -> quantidade_biomas_seeg
07 -> biomas_presentes_seeg
08 -> quantidade_biomas_ano
09 -> biomas_completos
10 -> biomas_parciais
11 -> biomas_nao_captados
12 -> status_dado_seeg
13 -> emissao_direta_co2e_gwp_ar6_soma_disponivel_t
14 -> emissao_indireta_co2e_gwp_ar6_soma_disponivel_t
15 -> emissao_total_co2e_gwp_ar6_soma_disponivel_t
16 -> emissao_total_co2e_gwp_ar6_t
17 -> emissao_direta_n2o_soma_disponivel_t
18 -> emissao_indireta_n2o_soma_disponivel_t
19 -> emissao_total_n2o_soma_disponivel_t
20 -> emissao_total_n2o_t
21 -> fonte
22 -> setor_emissao
23 -> categoria_emissao
24 -> subcategoria_emissao
25 -> gas_origem
26 -> metrica_co2e

COLUNAS BRLUC
01 -> codigo_ibge
02 -> municipio
03 -> estado
04 -> uf
05 -> regiao
06 -> cultura
07 -> periodo_inicio_brluc
08 -> periodo_fim_brluc
09 -> area_origem_temporaria_ha
10 -> area_origem_soja_ha
11 -> area_origem_cana_ha
12 -> area_origem_cultura_permanente_ha
1

In [9]:
# ============================================================
# CONSISTÊNCIA TERRITORIAL — BASE PRINCIPAL × SEEG
# ============================================================

COLUNAS_TERRITORIAIS_CANDIDATAS = [
    "municipio",
    "uf",
    "regiao",
    "cultura"
]


colunas_comuns_territoriais = [
    coluna
    for coluna in COLUNAS_TERRITORIAIS_CANDIDATAS
    if (
        coluna in base_agroambiental.columns
        and
        coluna in seeg.columns
    )
]


controle_territorial_seeg = (
    base_agroambiental[
        [
            "codigo_ibge",
            "ano"
        ]
        +
        colunas_comuns_territoriais
    ]
    .merge(
        seeg[
            [
                "codigo_ibge",
                "ano"
            ]
            +
            colunas_comuns_territoriais
        ],
        on=[
            "codigo_ibge",
            "ano"
        ],
        how="left",
        validate="one_to_one",
        suffixes=(
            "_base",
            "_seeg"
        )
    )
)


print("=" * 70)
print("CONSISTÊNCIA BASE × SEEG")
print("=" * 70)


for coluna in colunas_comuns_territoriais:

    coluna_base = (
        f"{coluna}_base"
    )

    coluna_seeg = (
        f"{coluna}_seeg"
    )

    divergencias = (
        controle_territorial_seeg[
            coluna_base
        ]
        .astype("string")
        .str.strip()
        !=
        controle_territorial_seeg[
            coluna_seeg
        ]
        .astype("string")
        .str.strip()
    )


    print(
        coluna,
        "-> divergências:",
        divergencias.sum()
    )

CONSISTÊNCIA BASE × SEEG
municipio -> divergências: 0
uf -> divergências: 0
cultura -> divergências: 8674


In [10]:
# ============================================================
# AUDITORIA DA VARIÁVEL CULTURA
# BASE PRINCIPAL × SEEG
# ============================================================

print("=" * 70)
print("CULTURA — BASE PRINCIPAL")
print("=" * 70)

display(
    base_agroambiental[
        "cultura"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\n" + "=" * 70
)

print(
    "CULTURA — SEEG"
)

print("=" * 70)

display(
    seeg[
        "cultura"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nValores únicos — Base:"
)

print(
    base_agroambiental[
        "cultura"
    ]
    .unique()
)


print(
    "\nValores únicos — SEEG:"
)

print(
    seeg[
        "cultura"
    ]
    .unique()
)

CULTURA — BASE PRINCIPAL


cultura
soja    8674
Name: count, dtype: int64


CULTURA — SEEG


cultura
Soja    9948
Name: count, dtype: int64


Valores únicos — Base:
['soja']

Valores únicos — SEEG:
['Soja']


In [11]:
# ============================================================
# CONSISTÊNCIA TERRITORIAL — BASE PRINCIPAL × BRLUC
# ============================================================

base_municipios = (
    base_agroambiental[
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao",
            "cultura"
        ]
    ]
    .drop_duplicates(
        subset=[
            "codigo_ibge"
        ]
    )
    .copy()
)


controle_territorial_brluc = (
    base_municipios
    .merge(
        brluc[
            [
                "codigo_ibge",
                "municipio",
                "uf",
                "regiao",
                "cultura"
            ]
        ],
        on="codigo_ibge",
        how="left",
        validate="one_to_one",
        suffixes=(
            "_base",
            "_brluc"
        )
    )
)


print("=" * 70)
print("CONSISTÊNCIA BASE × BRLUC")
print("=" * 70)


for coluna in [
    "municipio",
    "uf",
    "regiao",
    "cultura"
]:

    coluna_base = (
        f"{coluna}_base"
    )

    coluna_brluc = (
        f"{coluna}_brluc"
    )


    divergencias = (
        controle_territorial_brluc[
            coluna_base
        ]
        .astype("string")
        .str.strip()
        !=
        controle_territorial_brluc[
            coluna_brluc
        ]
        .astype("string")
        .str.strip()
    )


    print(
        coluna,
        "-> divergências:",
        divergencias.sum()
    )

CONSISTÊNCIA BASE × BRLUC
municipio -> divergências: 1
uf -> divergências: 0
regiao -> divergências: 0
cultura -> divergências: 1505


In [12]:
# ============================================================
# DEFINIÇÃO DAS COLUNAS DE INTEGRAÇÃO
# ============================================================

# ------------------------------------------------------------
# SEEG
# Chaves ficam.
# Identificadores redundantes não entram novamente.
# ------------------------------------------------------------

COLUNAS_SEEG_EXCLUIR = [
    "municipio",
    "uf",
    "cultura"
]


COLUNAS_SEEG_INTEGRAR = [
    coluna
    for coluna in seeg.columns
    if coluna not in COLUNAS_SEEG_EXCLUIR
]


# ------------------------------------------------------------
# BRLUC
# codigo_ibge fica como chave.
# Campos territoriais já existem na base principal.
# fonte será renomeada depois para não conflitar.
# ------------------------------------------------------------

COLUNAS_BRLUC_EXCLUIR = [
    "municipio",
    "estado",
    "uf",
    "regiao",
    "cultura"
]


COLUNAS_BRLUC_INTEGRAR = [
    coluna
    for coluna in brluc.columns
    if coluna not in COLUNAS_BRLUC_EXCLUIR
]


print("=" * 70)
print("SEEG — COLUNAS QUE ENTRARÃO")
print("=" * 70)

print(
    "Quantidade:",
    len(
        COLUNAS_SEEG_INTEGRAR
    )
)

print(
    COLUNAS_SEEG_INTEGRAR
)


print(
    "\n" + "=" * 70
)

print(
    "BRLUC — COLUNAS QUE ENTRARÃO"
)

print("=" * 70)

print(
    "Quantidade:",
    len(
        COLUNAS_BRLUC_INTEGRAR
    )
)

print(
    COLUNAS_BRLUC_INTEGRAR
)

SEEG — COLUNAS QUE ENTRARÃO
Quantidade: 23
['codigo_ibge', 'ano', 'quantidade_biomas_seeg', 'biomas_presentes_seeg', 'quantidade_biomas_ano', 'biomas_completos', 'biomas_parciais', 'biomas_nao_captados', 'status_dado_seeg', 'emissao_direta_co2e_gwp_ar6_soma_disponivel_t', 'emissao_indireta_co2e_gwp_ar6_soma_disponivel_t', 'emissao_total_co2e_gwp_ar6_soma_disponivel_t', 'emissao_total_co2e_gwp_ar6_t', 'emissao_direta_n2o_soma_disponivel_t', 'emissao_indireta_n2o_soma_disponivel_t', 'emissao_total_n2o_soma_disponivel_t', 'emissao_total_n2o_t', 'fonte', 'setor_emissao', 'categoria_emissao', 'subcategoria_emissao', 'gas_origem', 'metrica_co2e']

BRLUC — COLUNAS QUE ENTRARÃO
Quantidade: 69
['codigo_ibge', 'periodo_inicio_brluc', 'periodo_fim_brluc', 'area_origem_temporaria_ha', 'area_origem_soja_ha', 'area_origem_cana_ha', 'area_origem_cultura_permanente_ha', 'area_origem_pastagem_ha', 'area_origem_floresta_plantada_ha', 'area_origem_natural_ha', 'area_soja_t1_brluc_ha', 'area_soja_persiste

In [13]:
# ============================================================
# AUDITORIA DO ÚNICO MUNICÍPIO DIVERGENTE
# BASE PRINCIPAL × BRLUC
# ============================================================

divergencia_municipio_brluc = (
    controle_territorial_brluc[
        controle_territorial_brluc[
            "municipio_base"
        ]
        .astype("string")
        .str.strip()
        !=
        controle_territorial_brluc[
            "municipio_brluc"
        ]
        .astype("string")
        .str.strip()
    ]
    [
        [
            "codigo_ibge",
            "municipio_base",
            "municipio_brluc",
            "uf_base",
            "regiao_base"
        ]
    ]
)


print("=" * 70)
print("DIVERGÊNCIA DE NOME MUNICIPAL — BASE × BRLUC")
print("=" * 70)

print(
    "\nQuantidade:"
)

print(
    len(
        divergencia_municipio_brluc
    )
)


display(
    divergencia_municipio_brluc
)

DIVERGÊNCIA DE NOME MUNICIPAL — BASE × BRLUC

Quantidade:
1


,codigo_ibge,municipio_base,municipio_brluc,uf_base,regiao_base
1247,5107800,Santo Antônio de Leverger,Santo Antônio do Leverger,MT,Centro-Oeste


In [14]:
# ============================================================
# PREPARAÇÃO DAS FONTES PARA O MERGE
# ============================================================

# ------------------------------------------------------------
# SEEG
# ------------------------------------------------------------

seeg_integracao = (
    seeg[
        COLUNAS_SEEG_INTEGRAR
    ]
    .copy()
)


seeg_integracao = (
    seeg_integracao
    .rename(
        columns={
            "fonte":
                "fonte_seeg"
        }
    )
)


# ------------------------------------------------------------
# BRLUC
# ------------------------------------------------------------

brluc_integracao = (
    brluc[
        COLUNAS_BRLUC_INTEGRAR
    ]
    .copy()
)


brluc_integracao = (
    brluc_integracao
    .rename(
        columns={
            "fonte":
                "fonte_brluc"
        }
    )
)


print("=" * 70)
print("BASES PREPARADAS PARA INTEGRAÇÃO")
print("=" * 70)


print(
    "\nSEEG:"
)

print(
    seeg_integracao.shape
)


print(
    "Duplicatas codigo_ibge + ano:",
    seeg_integracao
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


print(
    "\nBRLUC:"
)

print(
    brluc_integracao.shape
)


print(
    "Duplicatas codigo_ibge:",
    brluc_integracao
    .duplicated(
        subset=[
            "codigo_ibge"
        ]
    )
    .sum()
)


print(
    "\nColuna fonte_seeg existe:",
    "fonte_seeg"
    in
    seeg_integracao.columns
)


print(
    "Coluna fonte_brluc existe:",
    "fonte_brluc"
    in
    brluc_integracao.columns
)

BASES PREPARADAS PARA INTEGRAÇÃO

SEEG:
(9948, 23)
Duplicatas codigo_ibge + ano: 0

BRLUC:
(1658, 69)
Duplicatas codigo_ibge: 0

Coluna fonte_seeg existe: True
Coluna fonte_brluc existe: True


In [15]:
# ============================================================
# MERGE 1 — BASE AGROAMBIENTAL + SEEG
# ============================================================

base_com_seeg = (
    base_agroambiental
    .merge(
        seeg_integracao,
        on=[
            "codigo_ibge",
            "ano"
        ],
        how="left",
        validate="one_to_one",
        indicator=True
    )
)


print("=" * 70)
print("MERGE — BASE AGROAMBIENTAL + SEEG")
print("=" * 70)


print(
    "\nDimensão antes de remover _merge:"
)

print(
    base_com_seeg.shape
)


print(
    "\nStatus do merge:"
)

display(
    base_com_seeg[
        "_merge"
    ]
    .value_counts()
)


print(
    "\nLinhas:"
)

print(
    len(
        base_com_seeg
    )
)


print(
    "\nMunicípios:"
)

print(
    base_com_seeg[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas codigo_ibge + ano:"
)

print(
    base_com_seeg
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


# ------------------------------------------------------------
# Remover indicador somente após validar
# ------------------------------------------------------------

base_com_seeg = (
    base_com_seeg
    .drop(
        columns=[
            "_merge"
        ]
    )
)


print(
    "\nDimensão final:"
)

print(
    base_com_seeg.shape
)

MERGE — BASE AGROAMBIENTAL + SEEG

Dimensão antes de remover _merge:
(8674, 67)

Status do merge:


_merge
both          8674
left_only        0
right_only       0
Name: count, dtype: int64


Linhas:
8674

Municípios:
1505

Duplicatas codigo_ibge + ano:
0

Dimensão final:
(8674, 66)


In [16]:
# ============================================================
# MERGE 2 — BASE + SEEG + BRLUC
# ============================================================

base_final_integrada = (
    base_com_seeg
    .merge(
        brluc_integracao,
        on="codigo_ibge",
        how="left",
        validate="many_to_one",
        indicator=True
    )
)


print("=" * 70)
print("MERGE — BASE + SEEG + BRLUC")
print("=" * 70)


print(
    "\nDimensão antes de remover _merge:"
)

print(
    base_final_integrada.shape
)


print(
    "\nStatus do merge:"
)

display(
    base_final_integrada[
        "_merge"
    ]
    .value_counts()
)


print(
    "\nLinhas:"
)

print(
    len(
        base_final_integrada
    )
)


print(
    "\nMunicípios:"
)

print(
    base_final_integrada[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas codigo_ibge + ano:"
)

print(
    base_final_integrada
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


# ------------------------------------------------------------
# Remover indicador após validação
# ------------------------------------------------------------

base_final_integrada = (
    base_final_integrada
    .drop(
        columns=[
            "_merge"
        ]
    )
)


print(
    "\nDimensão final:"
)

print(
    base_final_integrada.shape
)

MERGE — BASE + SEEG + BRLUC

Dimensão antes de remover _merge:
(8674, 135)

Status do merge:


_merge
both          8674
left_only        0
right_only       0
Name: count, dtype: int64


Linhas:
8674

Municípios:
1505

Duplicatas codigo_ibge + ano:
0

Dimensão final:
(8674, 134)


In [17]:
# ============================================================
# CONTROLE — BRLUC COMO CAMADA ESTÁTICA MUNICIPAL
# ============================================================

COLUNAS_BRLUC_CONTROLE_ESTATICO = [
    "periodo_inicio_brluc",
    "periodo_fim_brluc",
    "area_conversao_para_soja_2000_2019_ha",
    "emissao_absoluta_co2_t_ano",
    "taxa_emissao_co2_t_ha_ano",
    "soc_t0_t_c_ha",
    "soc_t1_t_c_ha",
    "ctotal_t0_t_c_ha",
    "ctotal_t1_t_c_ha"
]


print("=" * 70)
print("CONTROLE DE ESTABILIDADE — BRLUC")
print("=" * 70)


for coluna in COLUNAS_BRLUC_CONTROLE_ESTATICO:

    max_valores_por_municipio = (
        base_final_integrada
        .groupby(
            "codigo_ibge"
        )[
            coluna
        ]
        .nunique(
            dropna=False
        )
        .max()
    )

    print(
        coluna,
        "-> máximo de valores distintos por município:",
        max_valores_por_municipio
    )

CONTROLE DE ESTABILIDADE — BRLUC
periodo_inicio_brluc -> máximo de valores distintos por município: 1
periodo_fim_brluc -> máximo de valores distintos por município: 1
area_conversao_para_soja_2000_2019_ha -> máximo de valores distintos por município: 1
emissao_absoluta_co2_t_ano -> máximo de valores distintos por município: 1
taxa_emissao_co2_t_ha_ano -> máximo de valores distintos por município: 1
soc_t0_t_c_ha -> máximo de valores distintos por município: 1
soc_t1_t_c_ha -> máximo de valores distintos por município: 1
ctotal_t0_t_c_ha -> máximo de valores distintos por município: 1
ctotal_t1_t_c_ha -> máximo de valores distintos por município: 1


In [18]:
# ============================================================
# AUDITORIA DE COMPLETUDE — BASE FINAL
# ============================================================

print("=" * 70)
print("COMPLETUDE — BASE FINAL INTEGRADA")
print("=" * 70)


# ------------------------------------------------------------
# SEEG
# ------------------------------------------------------------

print(
    "\nStatus SEEG:"
)

display(
    base_final_integrada[
        "status_dado_seeg"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    "\nEmissão total disponível — NULL:"
)

print(
    base_final_integrada[
        "emissao_total_co2e_gwp_ar6_soma_disponivel_t"
    ]
    .isna()
    .sum()
)


print(
    "\nEmissão total completa — NULL:"
)

print(
    base_final_integrada[
        "emissao_total_co2e_gwp_ar6_t"
    ]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# BRLUC
# ------------------------------------------------------------

print(
    "\nBRLUC — emissão absoluta NULL:"
)

print(
    base_final_integrada[
        "emissao_absoluta_co2_t_ano"
    ]
    .isna()
    .sum()
)


print(
    "\nBRLUC — SOC t0 NULL:"
)

print(
    base_final_integrada[
        "soc_t0_t_c_ha"
    ]
    .isna()
    .sum()
)


print(
    "\nBRLUC — SOC t1 NULL:"
)

print(
    base_final_integrada[
        "soc_t1_t_c_ha"
    ]
    .isna()
    .sum()
)


print(
    "\nBRLUC — percentual de conversão NULL:"
)

print(
    base_final_integrada[
        "percentual_conversao_para_soja_pct"
    ]
    .isna()
    .sum()
)

COMPLETUDE — BASE FINAL INTEGRADA

Status SEEG:


status_dado_seeg
completo               8659
parcial_nao_captado      15
Name: count, dtype: int64


Emissão total disponível — NULL:
0

Emissão total completa — NULL:
15

BRLUC — emissão absoluta NULL:
0

BRLUC — SOC t0 NULL:
0

BRLUC — SOC t1 NULL:
0

BRLUC — percentual de conversão NULL:
84


In [19]:
# ============================================================
# AUDITORIA DOS REGISTROS PARCIAIS — SEEG
# ============================================================

seeg_parciais_final = (
    base_final_integrada[
        base_final_integrada[
            "status_dado_seeg"
        ]
        .eq(
            "parcial_nao_captado"
        )
    ]
    [
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "ano",

            "quantidade_biomas_seeg",
            "quantidade_biomas_ano",
            "biomas_completos",
            "biomas_parciais",
            "biomas_nao_captados",

            "emissao_total_co2e_gwp_ar6_soma_disponivel_t",
            "emissao_total_co2e_gwp_ar6_t"
        ]
    ]
    .sort_values(
        by=[
            "uf",
            "municipio",
            "ano"
        ]
    )
)


print("=" * 70)
print("REGISTROS PARCIAIS — SEEG")
print("=" * 70)


print(
    "\nQuantidade:"
)

print(
    len(
        seeg_parciais_final
    )
)


print(
    "\nMunicípios envolvidos:"
)

print(
    seeg_parciais_final[
        "codigo_ibge"
    ]
    .nunique()
)


display(
    seeg_parciais_final
)

REGISTROS PARCIAIS — SEEG

Quantidade:
15

Municípios envolvidos:
3


,codigo_ibge,municipio,uf,ano,quantidade_biomas_seeg,quantidade_biomas_ano,biomas_completos,biomas_parciais,biomas_nao_captados,emissao_total_co2e_gwp_ar6_soma_disponivel_t,emissao_total_co2e_gwp_ar6_t
4063,4305132,Cerro Branco,RS,2019,2,2,1,0,1,346.61,NaN
4064,4305132,Cerro Branco,RS,2020,2,2,1,0,1,205.12,NaN
4065,4305132,Cerro Branco,RS,2021,2,2,1,0,1,407.39,NaN
5236,4314803,Portão,RS,2019,2,2,1,0,1,45.58,NaN
5237,4314803,Portão,RS,2020,2,2,1,0,1,46.16,NaN
5238,4314803,Portão,RS,2021,2,2,1,0,1,68.37,NaN
5239,4314803,Portão,RS,2022,2,2,1,0,1,141.02,NaN
5240,4314803,Portão,RS,2023,2,2,1,0,1,94.02,NaN
5241,4314803,Portão,RS,2024,2,2,1,0,1,106.17,NaN
6038,4322608,Venâncio Aires,RS,2019,2,2,1,0,1,1282.00,NaN


In [20]:
# ============================================================
# CONTROLES MATEMÁTICOS — SEEG NA BASE FINAL
# ============================================================

dif_co2e_disponivel = (
    (
        base_final_integrada[
            "emissao_direta_co2e_gwp_ar6_soma_disponivel_t"
        ]
        +
        base_final_integrada[
            "emissao_indireta_co2e_gwp_ar6_soma_disponivel_t"
        ]
    )
    -
    base_final_integrada[
        "emissao_total_co2e_gwp_ar6_soma_disponivel_t"
    ]
).abs()


dif_n2o_disponivel = (
    (
        base_final_integrada[
            "emissao_direta_n2o_soma_disponivel_t"
        ]
        +
        base_final_integrada[
            "emissao_indireta_n2o_soma_disponivel_t"
        ]
    )
    -
    base_final_integrada[
        "emissao_total_n2o_soma_disponivel_t"
    ]
).abs()


print("=" * 70)
print("CONTROLES MATEMÁTICOS — SEEG")
print("=" * 70)


print(
    "\nMaior diferença CO2e direto + indireto:"
)

print(
    dif_co2e_disponivel.max()
)


print(
    "Falhas > 0.000001:"
)

print(
    (
        dif_co2e_disponivel
        > 0.000001
    )
    .sum()
)


print(
    "\nMaior diferença N2O direto + indireto:"
)

print(
    dif_n2o_disponivel.max()
)


print(
    "Falhas > 0.000001:"
)

print(
    (
        dif_n2o_disponivel
        > 0.000001
    )
    .sum()
)


print(
    "\nRegistros completos com emissão total completa NULL:"
)

print(
    (
        base_final_integrada[
            "status_dado_seeg"
        ]
        .eq(
            "completo"
        )
        &
        base_final_integrada[
            "emissao_total_co2e_gwp_ar6_t"
        ]
        .isna()
    )
    .sum()
)


print(
    "\nRegistros parciais com emissão total completa preenchida:"
)

print(
    (
        base_final_integrada[
            "status_dado_seeg"
        ]
        .eq(
            "parcial_nao_captado"
        )
        &
        base_final_integrada[
            "emissao_total_co2e_gwp_ar6_t"
        ]
        .notna()
    )
    .sum()
)

CONTROLES MATEMÁTICOS — SEEG

Maior diferença CO2e direto + indireto:
2.9103830456733704e-11
Falhas > 0.000001:
0

Maior diferença N2O direto + indireto:
1.1368683772161603e-13
Falhas > 0.000001:
0

Registros completos com emissão total completa NULL:
0

Registros parciais com emissão total completa preenchida:
0


In [21]:
# ============================================================
# CONTROLES MATEMÁTICOS — BRLUC NA BASE FINAL
# ============================================================

# ------------------------------------------------------------
# Ctotal = SOC + Cveg
# ------------------------------------------------------------

dif_brluc_ctotal_t0 = (
    base_final_integrada[
        "ctotal_t0_t_c_ha"
    ]
    -
    (
        base_final_integrada[
            "soc_t0_t_c_ha"
        ]
        +
        base_final_integrada[
            "cveg_t0_t_c_ha"
        ]
    )
).abs()


dif_brluc_ctotal_t1 = (
    base_final_integrada[
        "ctotal_t1_t_c_ha"
    ]
    -
    (
        base_final_integrada[
            "soc_t1_t_c_ha"
        ]
        +
        base_final_integrada[
            "cveg_t1_t_c_ha"
        ]
    )
).abs()


# ------------------------------------------------------------
# Delta Ctotal = Delta SOC
# porque Cveg é constante entre as duas classes
# ------------------------------------------------------------

dif_delta_brluc = (
    base_final_integrada[
        "delta_ctotal_classes_brluc_t1_t0_t_c_ha"
    ]
    -
    base_final_integrada[
        "delta_soc_classes_brluc_t1_t0_t_c_ha"
    ]
).abs()


print("=" * 70)
print("CONTROLES MATEMÁTICOS — BRLUC")
print("=" * 70)


print(
    "\nFalhas Ctotal = SOC + Cveg | t0:"
)

print(
    (
        dif_brluc_ctotal_t0
        > 0.000001
    )
    .sum()
)


print(
    "\nFalhas Ctotal = SOC + Cveg | t1:"
)

print(
    (
        dif_brluc_ctotal_t1
        > 0.000001
    )
    .sum()
)


print(
    "\nFalhas Delta Ctotal = Delta SOC:"
)

print(
    (
        dif_delta_brluc
        > 0.000001
    )
    .sum()
)


print(
    "\nCveg t0 e t1 diferentes:"
)

print(
    (
        (
            base_final_integrada[
                "cveg_t0_t_c_ha"
            ]
            -
            base_final_integrada[
                "cveg_t1_t_c_ha"
            ]
        )
        .abs()
        > 0.000001
    )
    .sum()
)

CONTROLES MATEMÁTICOS — BRLUC

Falhas Ctotal = SOC + Cveg | t0:
0

Falhas Ctotal = SOC + Cveg | t1:
0

Falhas Delta Ctotal = Delta SOC:
0

Cveg t0 e t1 diferentes:
0


In [22]:
# ============================================================
# AUDITORIA GLOBAL DE NULL — BASE FINAL
# ============================================================

auditoria_null = (
    pd.DataFrame(
        {
            "coluna":
                base_final_integrada.columns,

            "quantidade_null":
                base_final_integrada
                .isna()
                .sum()
                .values,

            "tipo":
                base_final_integrada
                .dtypes
                .astype("string")
                .values
        }
    )
)


auditoria_null[
    "percentual_null"
] = (
    auditoria_null[
        "quantidade_null"
    ]
    /
    len(
        base_final_integrada
    )
    *
    100
)


auditoria_null_com_valores = (
    auditoria_null[
        auditoria_null[
            "quantidade_null"
        ]
        > 0
    ]
    .sort_values(
        by=[
            "quantidade_null",
            "coluna"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


print("=" * 70)
print("AUDITORIA GLOBAL DE NULL")
print("=" * 70)


print(
    "\nTotal de colunas:"
)

print(
    base_final_integrada.shape[1]
)


print(
    "\nColunas com pelo menos um NULL:"
)

print(
    len(
        auditoria_null_com_valores
    )
)


print(
    "\nTotal de células NULL:"
)

print(
    int(
        base_final_integrada
        .isna()
        .sum()
        .sum()
    )
)


display(
    auditoria_null_com_valores
)

AUDITORIA GLOBAL DE NULL

Total de colunas:
134

Colunas com pelo menos um NULL:
16

Total de células NULL:
789


,coluna,quantidade_null,tipo,percentual_null
0,taxa_emissao_co2_ic95_inf,103,float64,1.187457
1,taxa_emissao_co2_ic95_sup,103,float64,1.187457
2,taxa_emissao_co2_se,103,float64,1.187457
3,emissao_absoluta_co2_ic95_inf,84,float64,0.968411
4,emissao_absoluta_co2_ic95_sup,84,float64,0.968411
5,emissao_absoluta_co2_se,84,float64,0.968411
6,percentual_conversao_para_soja_pct,84,float64,0.968411
7,percentual_persistencia_soja_pct,84,float64,0.968411
8,emissao_total_co2e_gwp_ar6_t,15,float64,0.172931
9,emissao_total_n2o_t,15,float64,0.172931


In [23]:
# ============================================================
# NULL POR ORIGEM DOS DADOS
# ============================================================

COLUNAS_ORIGEM_BASE = list(
    base_agroambiental.columns
)


COLUNAS_ORIGEM_SEEG = [
    coluna
    for coluna in seeg_integracao.columns
    if coluna not in [
        "codigo_ibge",
        "ano"
    ]
]


COLUNAS_ORIGEM_BRLUC = [
    coluna
    for coluna in brluc_integracao.columns
    if coluna != "codigo_ibge"
]


def resumo_null_origem(
    dataframe,
    colunas,
    origem
):

    resultado = (
        dataframe[
            colunas
        ]
        .isna()
        .sum()
        .reset_index()
    )

    resultado.columns = [
        "coluna",
        "quantidade_null"
    ]

    resultado[
        "origem"
    ] = origem

    resultado[
        "percentual_null"
    ] = (
        resultado[
            "quantidade_null"
        ]
        /
        len(
            dataframe
        )
        *
        100
    )

    return (
        resultado[
            resultado[
                "quantidade_null"
            ]
            > 0
        ]
        .copy()
    )


null_base = resumo_null_origem(
    base_final_integrada,
    COLUNAS_ORIGEM_BASE,
    "Base PAM + Solo + INMET + Cobertura"
)


null_seeg = resumo_null_origem(
    base_final_integrada,
    COLUNAS_ORIGEM_SEEG,
    "SEEG"
)


null_brluc = resumo_null_origem(
    base_final_integrada,
    COLUNAS_ORIGEM_BRLUC,
    "BRLUC"
)


auditoria_null_por_origem = (
    pd.concat(
        [
            null_base,
            null_seeg,
            null_brluc
        ],
        ignore_index=True
    )
    .sort_values(
        by=[
            "origem",
            "quantidade_null",
            "coluna"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


print("=" * 70)
print("NULL POR ORIGEM")
print("=" * 70)


display(
    auditoria_null_por_origem
)

NULL POR ORIGEM


,coluna,quantidade_null,origem,percentual_null
0,taxa_emissao_co2_ic95_inf,103,BRLUC,1.187457
1,taxa_emissao_co2_ic95_sup,103,BRLUC,1.187457
2,taxa_emissao_co2_se,103,BRLUC,1.187457
3,emissao_absoluta_co2_ic95_inf,84,BRLUC,0.968411
4,emissao_absoluta_co2_ic95_sup,84,BRLUC,0.968411
5,emissao_absoluta_co2_se,84,BRLUC,0.968411
6,percentual_conversao_para_soja_pct,84,BRLUC,0.968411
7,percentual_persistencia_soja_pct,84,BRLUC,0.968411
8,aproveitamento_area_pct,5,Base PAM + Solo + INMET + Cobertura,0.057644
9,area_colhida_ha,5,Base PAM + Solo + INMET + Cobertura,0.057644


In [24]:
# ============================================================
# INTEGRIDADE TÉCNICA — BASE FINAL
# ============================================================

print("=" * 70)
print("INTEGRIDADE TÉCNICA — BASE FINAL")
print("=" * 70)


# ------------------------------------------------------------
# Dimensão e chave
# ------------------------------------------------------------

print(
    "\nDimensão:"
)

print(
    base_final_integrada.shape
)


print(
    "\nDuplicatas codigo_ibge + ano:"
)

print(
    base_final_integrada
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


print(
    "\nNULL em codigo_ibge:"
)

print(
    base_final_integrada[
        "codigo_ibge"
    ]
    .isna()
    .sum()
)


print(
    "\nNULL em ano:"
)

print(
    base_final_integrada[
        "ano"
    ]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# Colunas duplicadas
# ------------------------------------------------------------

print(
    "\nNomes de coluna duplicados:"
)

print(
    base_final_integrada
    .columns
    .duplicated()
    .sum()
)


# ------------------------------------------------------------
# Sufixos indesejados de merge
# ------------------------------------------------------------

colunas_com_sufixo_merge = [
    coluna
    for coluna in base_final_integrada.columns
    if (
        coluna.endswith("_x")
        or
        coluna.endswith("_y")
    )
]


print(
    "\nColunas _x ou _y:"
)

print(
    colunas_com_sufixo_merge
)


# ------------------------------------------------------------
# Valores infinitos
# ------------------------------------------------------------

colunas_numericas = (
    base_final_integrada
    .select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)


quantidade_inf = (
    np.isinf(
        base_final_integrada[
            colunas_numericas
        ]
    )
    .sum()
)


colunas_com_inf = (
    quantidade_inf[
        quantidade_inf
        > 0
    ]
)


print(
    "\nColunas com valores infinitos:"
)

if len(
    colunas_com_inf
) == 0:

    print(
        "Nenhuma"
    )

else:

    display(
        colunas_com_inf
    )


# ------------------------------------------------------------
# Universo temporal
# ------------------------------------------------------------

print(
    "\nAnos:"
)

print(
    sorted(
        base_final_integrada[
            "ano"
        ]
        .unique()
    )
)


print(
    "\nCultura:"
)

print(
    base_final_integrada[
        "cultura"
    ]
    .value_counts(
        dropna=False
    )
)

INTEGRIDADE TÉCNICA — BASE FINAL

Dimensão:
(8674, 134)

Duplicatas codigo_ibge + ano:
0

NULL em codigo_ibge:
0

NULL em ano:
0

Nomes de coluna duplicados:
0

Colunas _x ou _y:
[]

Colunas com valores infinitos:
Nenhuma

Anos:
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Cultura:
cultura
soja    8674
Name: count, dtype: int64


In [25]:
# ============================================================
# NULL BRLUC — LINHAS × MUNICÍPIOS
# ============================================================

COLUNAS_BRLUC_NULL_CONTROLE = [
    "percentual_conversao_para_soja_pct",
    "percentual_persistencia_soja_pct",

    "emissao_absoluta_co2_se",
    "emissao_absoluta_co2_ic95_inf",
    "emissao_absoluta_co2_ic95_sup",

    "taxa_emissao_co2_se",
    "taxa_emissao_co2_ic95_inf",
    "taxa_emissao_co2_ic95_sup"
]


print("=" * 70)
print("NULL BRLUC — LINHAS E MUNICÍPIOS")
print("=" * 70)


for coluna in COLUNAS_BRLUC_NULL_CONTROLE:

    mascara_null = (
        base_final_integrada[
            coluna
        ]
        .isna()
    )

    quantidade_linhas = (
        mascara_null
        .sum()
    )

    quantidade_municipios = (
        base_final_integrada
        .loc[
            mascara_null,
            "codigo_ibge"
        ]
        .nunique()
    )

    print(
        f"\n{coluna}"
    )

    print(
        "Linhas NULL:",
        quantidade_linhas
    )

    print(
        "Municípios distintos:",
        quantidade_municipios
    )

NULL BRLUC — LINHAS E MUNICÍPIOS

percentual_conversao_para_soja_pct
Linhas NULL: 84
Municípios distintos: 27

percentual_persistencia_soja_pct
Linhas NULL: 84
Municípios distintos: 27

emissao_absoluta_co2_se
Linhas NULL: 84
Municípios distintos: 27

emissao_absoluta_co2_ic95_inf
Linhas NULL: 84
Municípios distintos: 27

emissao_absoluta_co2_ic95_sup
Linhas NULL: 84
Municípios distintos: 27

taxa_emissao_co2_se
Linhas NULL: 103
Municípios distintos: 31

taxa_emissao_co2_ic95_inf
Linhas NULL: 103
Municípios distintos: 31

taxa_emissao_co2_ic95_sup
Linhas NULL: 103
Municípios distintos: 31


In [26]:
# ============================================================
# EXPORTAÇÃO — BASE AGROAMBIENTAL FINAL
# ============================================================

DIRETORIO_BASE_FINAL = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "integracao_agroambiental"
)


DIRETORIO_BASE_FINAL.mkdir(
    parents=True,
    exist_ok=True
)


ARQUIVO_BASE_FINAL = (
    DIRETORIO_BASE_FINAL
    / "base_agroambiental_final_soja_centro_oeste_sul_2019_2024.csv"
)


# ------------------------------------------------------------
# Preparar base
# ------------------------------------------------------------

base_final_exportacao = (
    base_final_integrada
    .copy()
)


base_final_exportacao[
    "codigo_ibge"
] = (
    base_final_exportacao[
        "codigo_ibge"
    ]
    .astype("string")
    .str.zfill(7)
)


base_final_exportacao = (
    base_final_exportacao
    .sort_values(
        by=[
            "uf",
            "codigo_ibge",
            "ano"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Exportar
# ------------------------------------------------------------

base_final_exportacao.to_csv(
    ARQUIVO_BASE_FINAL,
    index=False,
    encoding="utf-8-sig"
)


print("=" * 70)
print("EXPORTAÇÃO — BASE AGROAMBIENTAL FINAL")
print("=" * 70)


print(
    "\nArquivo:"
)

print(
    ARQUIVO_BASE_FINAL
)


print(
    "\nExiste:"
)

print(
    ARQUIVO_BASE_FINAL.exists()
)


print(
    "\nDimensão:"
)

print(
    base_final_exportacao.shape
)


print(
    "\nTamanho:"
)

print(
    f"{ARQUIVO_BASE_FINAL.stat().st_size / (1024 * 1024):.2f} MB"
)

EXPORTAÇÃO — BASE AGROAMBIENTAL FINAL

Arquivo:
C:\Users\Elaine Cristina\Downloads\agroesg-analytics_correcao\agroesg-analytics_correcao\data\databases_curated\integracao_agroambiental\base_agroambiental_final_soja_centro_oeste_sul_2019_2024.csv

Existe:
True

Dimensão:
(8674, 134)

Tamanho:
15.42 MB


In [27]:
# ============================================================
# VALIDAÇÃO PÓS-EXPORTAÇÃO — BASE FINAL
# ============================================================

base_final_recarregada = pd.read_csv(
    ARQUIVO_BASE_FINAL,
    dtype={
        "codigo_ibge": "string"
    },
    encoding="utf-8-sig"
)


base_final_recarregada[
    "codigo_ibge"
] = (
    base_final_recarregada[
        "codigo_ibge"
    ]
    .str.zfill(7)
)


print("=" * 70)
print("VALIDAÇÃO DO CSV FINAL")
print("=" * 70)


print(
    "\nDimensão:"
)

print(
    base_final_recarregada.shape
)


print(
    "\nMunicípios:"
)

print(
    base_final_recarregada[
        "codigo_ibge"
    ]
    .nunique()
)


print(
    "\nDuplicatas codigo_ibge + ano:"
)

print(
    base_final_recarregada
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)


print(
    "\nMesmo universo de chaves:"
)

print(
    set(
        zip(
            base_final_recarregada[
                "codigo_ibge"
            ],
            base_final_recarregada[
                "ano"
            ]
        )
    )
    ==
    set(
        zip(
            base_final_exportacao[
                "codigo_ibge"
            ],
            base_final_exportacao[
                "ano"
            ]
        )
    )
)


print(
    "\nTotal de NULL:"
)

print(
    int(
        base_final_recarregada
        .isna()
        .sum()
        .sum()
    )
)


print(
    "\nRegistros SEEG parciais:"
)

print(
    (
        base_final_recarregada[
            "status_dado_seeg"
        ]
        ==
        "parcial_nao_captado"
    )
    .sum()
)


print(
    "\nBRLUC — período:"
)

print(
    base_final_recarregada[
        "periodo_inicio_brluc"
    ]
    .unique(),
    "→",
    base_final_recarregada[
        "periodo_fim_brluc"
    ]
    .unique()
)

VALIDAÇÃO DO CSV FINAL

Dimensão:
(8674, 134)

Municípios:
1505

Duplicatas codigo_ibge + ano:
0

Mesmo universo de chaves:
True

Total de NULL:
789

Registros SEEG parciais:
15

BRLUC — período:
[2000] → [2019]


## Conclusão da integração agroambiental

A base agroambiental consolidada para soja foi construída a partir da
integração das seguintes fontes e camadas previamente processadas:

- IBGE/PAM — produção agrícola;
- MapBiomas Solo — indicadores de carbono do solo;
- INMET — indicadores climáticos municipalizados;
- MapBiomas Cobertura — uso e cobertura da terra;
- SEEG — emissões de N₂O associadas aos resíduos agrícolas da soja;
- Embrapa BRLUC — mudança de uso da terra, emissões de CO₂ e estoques
  de carbono.

### Resultado final

A base consolidada possui:

- 8.674 observações;
- 1.505 municípios;
- período anual de 2019 a 2024;
- 134 variáveis;
- nenhuma duplicidade na chave `codigo_ibge + ano`.

A cobertura do SEEG e do BRLUC sobre o universo analítico da PAM foi
de 100%.

### Valores ausentes

Foram preservados 789 valores ausentes, todos associados a situações
identificadas nas fontes originais:

- 5 observações agrícolas da PAM com variáveis ausentes;
- 15 observações SEEG classificadas como `parcial_nao_captado`;
- valores de incerteza e intervalos de confiança indisponíveis no BRLUC;
- percentuais BRLUC indefinidos quando o denominador da operação é zero.

Nenhum valor ausente foi artificialmente substituído por zero.

### Tratamento temporal do BRLUC

O BRLUC foi integrado pela chave municipal `codigo_ibge`.

Seus atributos são repetidos nas observações anuais apenas como
características estruturais dos municípios. Eles continuam
representando a transição BRLUC entre 2000 e 2019 e não devem ser
interpretados como valores recalculados para cada ano de 2019 a 2024.

### Interpretação das emissões

Os indicadores do SEEG utilizados nesta base representam especificamente
emissões de N₂O associadas aos resíduos agrícolas da soja em solos
manejados, diretas e indiretas, representadas em CO₂e pelo GWP-AR6.

Os indicadores do BRLUC representam resultados de mudança de uso da
terra e estoques de carbono conforme a metodologia BRLUC 2.1.

Nenhuma dessas variáveis deve ser interpretada diretamente como
quantidade de créditos de carbono gerados ou comercializáveis.